1. BaseTool — 추상 기본 클래스 (설계도)
2. 구체적인 Tool 3개 — Calculator, StringProcessor, Dictionary
3. ToolRegistry — 도구 보관함
4. Tool 체이닝 — 도구를 연결해서 실행
5. ReAct 시뮬레이션 통합 — 전부 합쳐서 실행

In [1]:
# ─────────────────────────────────────────
# abc 모듈 : 추상 클래스를 만들기 위한 모듈
# ABC      : 추상 클래스의 부모 클래스
# abstractmethod : 반드시 구현해야 하는 메서드를 지정하는 데코레이터
# ─────────────────────────────────────────
from abc import ABC, abstractmethod

# ─────────────────────────────────────────
# 타입 힌트용 모듈
# Any      : 어떤 타입이든 허용
# Dict     : 딕셔너리 타입
# Optional : None이 될 수도 있는 타입
# ─────────────────────────────────────────
from typing import Any, Dict, Optional
import json  # 스키마를 보기 좋게 출력할 때 사용


# ─────────────────────────────────────────
# BaseTool : 모든 Tool이 반드시 따라야 하는 설계도
# ABC를 상속받아서 추상 클래스로 만듦
# 이 클래스 자체는 인스턴스 생성 불가
# → 반드시 상속받아서 execute()를 구현해야만 사용 가능
# ─────────────────────────────────────────
class BaseTool(ABC):

    # ─────────────────────────────────────────
    # name        : Tool 이름 (Registry에서 찾을 때 사용하는 키)
    # description : Tool 설명 (LLM이 어떤 Tool을 쓸지 판단하는 근거)
    # ─────────────────────────────────────────
    def __init__(self, name: str, description: str):
        self.name = name
        self.description = description

    # ─────────────────────────────────────────
    # @abstractmethod : 자식 클래스에서 반드시 구현 강제
    # 구현 안 하면 인스턴스 생성 자체가 불가능
    # 모든 Tool은 문자열을 받아서 문자열을 반환하는 형태로 통일
    # → Registry가 어떤 Tool이든 동일한 방식으로 실행 가능
    # ─────────────────────────────────────────
    @abstractmethod
    def execute(self, input_text: str) -> str:
        pass

    # ─────────────────────────────────────────
    # Tool의 정보를 JSON Schema 형태로 반환
    # LLM에게 프롬프트로 넘길 때 이 스키마를 사용
    # _get_parameters()는 자식 클래스에서 오버라이드 가능
    # ─────────────────────────────────────────
    def get_schema(self) -> Dict:
        return {
            'name': self.name,
            'description': self.description,
            'parameters': self._get_parameters()
        }

    # ─────────────────────────────────────────
    # 기본 파라미터 스키마
    # 자식 클래스에서 더 구체적으로 오버라이드 가능
    # ─────────────────────────────────────────
    def _get_parameters(self) -> Dict:
        return {
            'type': 'string',
            'description': '입력 텍스트'
        }

    # ─────────────────────────────────────────
    # __repr__ : print(tool) 했을 때 출력되는 형태 정의
    # 없으면 <__main__.CalculatorTool object at 0x...> 같은 형태로 출력
    # ─────────────────────────────────────────
    def __repr__(self):
        return f'Tool(name={self.name})'


# ─────────────────────────────────────────
# 추상 클래스 테스트
# BaseTool을 직접 인스턴스화하면 에러 발생 확인
# ─────────────────────────────────────────
try:
    tool = BaseTool("test", "test")  # 에러 발생!
except TypeError as e:
    print(f"예상된 에러: {e}")
    # 출력 → Can't instantiate abstract class BaseTool
    #         with abstract method execute

예상된 에러: Can't instantiate abstract class BaseTool with abstract method execute


In [2]:
# ─────────────────────────────────────────
# CalculatorTool : 수식 계산 도구
# BaseTool을 상속받아서 execute()를 구현
# ─────────────────────────────────────────
class CalculatorTool(BaseTool):

    def __init__(self):
        # ─────────────────────────────────────────
        # super().__init__() : 부모 클래스(BaseTool)의
        # __init__을 호출해서 name, description 설정
        # 직접 self.name = ... 안 해도 되는 이유
        # ─────────────────────────────────────────
        super().__init__(
            name='Calculator',
            description='수학 수식을 계산합니다. 사칙연산 거듭제곱 나머지연산 등을 지원합니다.'
        )

    def execute(self, input_text: str) -> str:
        try:
            # ─────────────────────────────────────────
            # eval()은 강력하지만 보안 위험이 있음
            # 예: eval("import os; os.system('rm -rf /')")
            # → 허용된 문자만 통과시켜서 보안 처리
            # set() : 중복 없는 문자 집합으로 만들어서 빠르게 비교
            # ─────────────────────────────────────────
            allowed_chars = set('0123456789+-*/.() ')
            if not all(c in allowed_chars for c in input_text):
                return f"오류: 허용되지 않는 문자가 포함되어 있습니다."

            result = eval(input_text)  # 안전한 수식만 eval()로 실행
            return f"계산 결과: {result}"

        except Exception as e:
            return f"계산 오류: {str(e)}"


# ─────────────────────────────────────────
# StringProcessorTool : 문자열 분석 도구
# 문자 수 / 단어 수 / 줄 수를 분석
# ─────────────────────────────────────────
class StringProcessorTool(BaseTool):

    def __init__(self):
        super().__init__(
            name="StringProcessor",
            description="문자열의 길이, 단어 수, 줄 수 등을 분석합니다."
        )

    def execute(self, input_text: str) -> str:
        # ─────────────────────────────────────────
        # len()        : 전체 문자 수
        # .split()     : 공백 기준으로 분리 → 단어 수
        # .splitlines(): 줄바꿈 기준으로 분리 → 줄 수
        # ─────────────────────────────────────────
        char_count = len(input_text)
        word_count = len(input_text.split())
        line_count = len(input_text.splitlines())

        return (f"문자 수: {char_count}, "
                f"단어 수: {word_count}, "
                f"줄 수: {line_count}")


# ─────────────────────────────────────────
# DictionaryTool : 용어 검색 도구
# 미리 정의된 dict에서 용어를 찾아서 반환
# ─────────────────────────────────────────
class DictionaryTool(BaseTool):

    def __init__(self):
        super().__init__(
            name="Dictionary",
            description="내장 사전에서 용어의 정의를 검색합니다."
        )
        # ─────────────────────────────────────────
        # 실제 서비스에서는 DB나 외부 API로 대체될 부분
        # 지금은 하드코딩된 dict로 구현
        # ─────────────────────────────────────────
        self.entries = {
            "인공지능": "인간의 학습, 추론, 지각 능력을 컴퓨터로 구현하는 기술",
            "머신러닝": "데이터로부터 패턴을 학습하여 예측이나 결정을 수행하는 AI의 하위 분야",
            "딥러닝": "다층 신경망을 사용하여 복잡한 패턴을 학습하는 머신러닝의 하위 분야",
            "자연어처리": "컴퓨터가 인간의 언어를 이해하고 생성하는 AI 기술",
            "강화학습": "환경과의 상호작용을 통해 보상을 최대화하는 행동을 학습하는 방법",
            "트랜스포머": "어텐션 메커니즘 기반의 신경망 아키텍처로, 현대 NLP의 기반",
            "LLM": "대규모 텍스트 데이터로 사전 학습된 거대 언어 모델",
            "프롬프트": "LLM에 입력되는 지시문 또는 질의문",
            "ReAct": "추론(Reasoning)과 행동(Acting)을 결합한 프롬프팅 기법"
        }

    def execute(self, input_text: str) -> str:
        term = input_text.strip()  # 앞뒤 공백 제거

        # ─────────────────────────────────────────
        # 1순위 : 정확히 일치하는 키가 있으면 바로 반환
        # ─────────────────────────────────────────
        if term in self.entries:
            return f"{term}: {self.entries[term]}"

        # ─────────────────────────────────────────
        # 2순위 : 부분 일치 검색
        # term이 키에 포함되거나 키가 term에 포함되는 경우
        # 예: "딥" 검색 → "딥러닝" 매칭
        # ─────────────────────────────────────────
        matches = [k for k in self.entries if term in k or k in term]
        if matches:
            results = [f"{k}: {self.entries[k]}" for k in matches]
            return "관련 항목:\n" + "\n".join(results)

        return f"'{term}'에 대한 정보를 찾을 수 없습니다."


# ─────────────────────────────────────────
# 3개 Tool 테스트
# ─────────────────────────────────────────
calc = CalculatorTool()
print(f"스키마: {json.dumps(calc.get_schema(), ensure_ascii=False, indent=2)}")
print(f"정상 계산: {calc.execute('(15+25)*3')}")       # → 계산 결과: 120
print(f"보안 차단: {calc.execute('import os')}")        # → 허용되지 않는 문자

string_proc = StringProcessorTool()
print(f"문자열 분석: {string_proc.execute('ReAct는 대세입니다.')}")

dict_tool = DictionaryTool()
print(f"정확 매칭: {dict_tool.execute('딥러닝')}")
print(f"부분 매칭: {dict_tool.execute('딥')}")           # → 딥러닝 매칭
print(f"매칭 없음: {dict_tool.execute('블록체인')}")

스키마: {
  "name": "Calculator",
  "description": "수학 수식을 계산합니다. 사칙연산 거듭제곱 나머지연산 등을 지원합니다.",
  "parameters": {
    "type": "string",
    "description": "입력 텍스트"
  }
}
정상 계산: 계산 결과: 120
보안 차단: 오류: 허용되지 않는 문자가 포함되어 있습니다.
문자열 분석: 문자 수: 13, 단어 수: 2, 줄 수: 1
정확 매칭: 딥러닝: 다층 신경망을 사용하여 복잡한 패턴을 학습하는 머신러닝의 하위 분야
부분 매칭: 관련 항목:
딥러닝: 다층 신경망을 사용하여 복잡한 패턴을 학습하는 머신러닝의 하위 분야
매칭 없음: '블록체인'에 대한 정보를 찾을 수 없습니다.


In [3]:
# ─────────────────────────────────────────
# ToolRegistry : 모든 Tool을 한 곳에서 등록/조회/실행
# LLM이 Action: Calculator[2**10] 을 출력하면
# 파서가 ("Calculator", "2**10") 추출
# → Registry가 "Calculator" 찾아서 실행
# ─────────────────────────────────────────
class ToolRegistry:

    def __init__(self):
        # ─────────────────────────────────────────
        # _tools : Tool을 저장하는 딕셔너리
        # key   = Tool 이름 (문자열)
        # value = Tool 객체 (BaseTool 자식 클래스)
        # _ 언더스코어 : 외부에서 직접 접근하지 말라는 관례
        # Dict[str, BaseTool] : 타입 힌트
        # ─────────────────────────────────────────
        self._tools: Dict[str, BaseTool] = {}

    # ─────────────────────────────────────────
    # Tool을 등록하는 메서드
    # tool.name을 key로 사용해서 딕셔너리에 저장
    # return self : 체이닝 가능
    # registry.register(A).register(B).register(C)
    # ─────────────────────────────────────────
    def register(self, tool: BaseTool) -> 'ToolRegistry':
        self._tools[tool.name] = tool
        return self

    # ─────────────────────────────────────────
    # 이름으로 Tool을 조회하는 메서드
    # .get() : 키가 없으면 None 반환 (KeyError 안 남)
    # Optional[BaseTool] : Tool 또는 None 반환
    # ─────────────────────────────────────────
    def get(self, name: str) -> Optional[BaseTool]:
        return self._tools.get(name)

    # ─────────────────────────────────────────
    # Tool을 이름으로 찾아서 실행하는 메서드
    # 1. 이름으로 Tool 조회
    # 2. 없으면 에러 메시지 반환 (예외 발생 X)
    # 3. 있으면 execute() 호출
    # 에러를 예외로 올리지 않고 문자열로 반환하는 이유
    # → Observation으로 LLM에게 그대로 넘기기 위해
    # ─────────────────────────────────────────
    def execute(self, tool_name: str, input_text: str) -> str:
        tool = self.get(tool_name)

        # Tool을 찾지 못한 경우
        # 사용 가능한 Tool 목록도 함께 알려줌
        if tool is None:
            return (f"오류: '{tool_name}' 도구를 찾을 수 없습니다. "
                    f"사용 가능한 도구: {list(self._tools.keys())}")
        try:
            return tool.execute(input_text)
        except Exception as e:
            return f"실행 오류: {e}"

    # ─────────────────────────────────────────
    # 등록된 모든 Tool 목록을 문자열로 반환
    # 4단계 프롬프트 템플릿에 넣을 때 사용
    # ─────────────────────────────────────────
    def list_tools(self) -> str:
        descriptions = []
        for name, tool in self._tools.items():
            descriptions.append(f"  - {name}: {tool.description}")
        return "사용 가능한 도구\n" + "\n".join(descriptions)

    # ─────────────────────────────────────────
    # 등록된 모든 Tool의 스키마를 리스트로 반환
    # LLM에게 한번에 모든 Tool 정보를 넘길 때 사용
    # ─────────────────────────────────────────
    def get_all_schemas(self) -> list:
        return [tool.get_schema() for tool in self._tools.values()]


# ─────────────────────────────────────────
# Registry 생성 + Tool 3개 등록
# ─────────────────────────────────────────
registry = ToolRegistry()

# 체이닝으로 한번에 등록
registry.register(CalculatorTool()) \
        .register(StringProcessorTool()) \
        .register(DictionaryTool())

# 등록된 Tool 목록 출력
print(registry.list_tools())

# ─────────────────────────────────────────
# Registry를 통한 실행 테스트
# ─────────────────────────────────────────
print("\n=== Registry 실행 테스트 ===")

# 정상 실행
print(registry.execute("Calculator", "2 ** 10"))       # → 계산 결과: 1024
print(registry.execute("Dictionary", "딥러닝"))         # → 딥러닝: ...

# 존재하지 않는 Tool 호출
print(registry.execute("UnknownTool", "test"))          # → 오류: 찾을 수 없습니다

# 전체 스키마 출력
print("\n=== 전체 스키마 ===")
print(json.dumps(registry.get_all_schemas(), ensure_ascii=False, indent=2))

사용 가능한 도구
  - Calculator: 수학 수식을 계산합니다. 사칙연산 거듭제곱 나머지연산 등을 지원합니다.
  - StringProcessor: 문자열의 길이, 단어 수, 줄 수 등을 분석합니다.
  - Dictionary: 내장 사전에서 용어의 정의를 검색합니다.

=== Registry 실행 테스트 ===
계산 결과: 1024
딥러닝: 다층 신경망을 사용하여 복잡한 패턴을 학습하는 머신러닝의 하위 분야
오류: 'UnknownTool' 도구를 찾을 수 없습니다. 사용 가능한 도구: ['Calculator', 'StringProcessor', 'Dictionary']

=== 전체 스키마 ===
[
  {
    "name": "Calculator",
    "description": "수학 수식을 계산합니다. 사칙연산 거듭제곱 나머지연산 등을 지원합니다.",
    "parameters": {
      "type": "string",
      "description": "입력 텍스트"
    }
  },
  {
    "name": "StringProcessor",
    "description": "문자열의 길이, 단어 수, 줄 수 등을 분석합니다.",
    "parameters": {
      "type": "string",
      "description": "입력 텍스트"
    }
  },
  {
    "name": "Dictionary",
    "description": "내장 사전에서 용어의 정의를 검색합니다.",
    "parameters": {
      "type": "string",
      "description": "입력 텍스트"
    }
  }
]


In [4]:
# ─────────────────────────────────────────
# tool_chain : 여러 Tool을 순서대로 실행하는 함수
# 이전 Tool의 결과를 다음 Tool의 입력으로 연결
#
# registry : ToolRegistry 인스턴스
# steps    : (tool_name, input_data) 튜플의 리스트
#            input_data가 일반 값이면 그대로 사용
#            input_data가 함수(callable)면 이전 결과를 인자로 받아서 사용
# ─────────────────────────────────────────
def tool_chain(registry, steps):

    print("=== Tool 체이닝 실행 ===")
    previous_result = None  # 이전 Tool의 결과를 저장하는 변수

    for i, (tool_name, input_data) in enumerate(steps, 1):

        # ─────────────────────────────────────────
        # callable() : 해당 객체가 함수처럼 호출 가능한지 확인
        # True  → lambda 또는 함수 → 이전 결과를 인자로 넘겨서 입력 생성
        # False → 일반 문자열     → 그대로 입력으로 사용
        #
        # 이렇게 나누는 이유 :
        # 어떤 스텝은 고정된 입력이 필요하고
        # 어떤 스텝은 이전 결과를 가공해서 입력으로 써야 하기 때문
        # ─────────────────────────────────────────
        if callable(input_data):
            actual_input = input_data(previous_result)  # 이전 결과를 인자로 전달
        else:
            actual_input = input_data  # 고정 입력값 그대로 사용

        print(f"\n[Chain Step {i}] {tool_name}")
        print(f"  입력: {actual_input}")

        # Registry를 통해 Tool 실행
        result = registry.execute(tool_name, actual_input)
        print(f"  출력: {result}")

        # 다음 스텝을 위해 현재 결과를 저장
        previous_result = result

    print(f"\n최종 결과: {previous_result}")
    return previous_result


# ─────────────────────────────────────────
# 체이닝 예시 1
# Calculator 결과 → StringProcessor 입력으로 연결
# lambda prev: prev → 이전 결과를 그대로 다음 입력으로 전달
# ─────────────────────────────────────────
print("[예시 1] 계산 결과를 문자열로 분석")
chain_steps_1 = [
    ("Calculator", "(100 + 200 + 300) / 3"),       # 고정 입력
    ("StringProcessor", lambda prev: prev),          # 이전 결과를 입력으로
]
tool_chain(registry, chain_steps_1)


# ─────────────────────────────────────────
# 체이닝 예시 2
# Dictionary 검색 → StringProcessor → Calculator
# 각 스텝이 독립적인 입력을 가질 수도 있음
# ─────────────────────────────────────────
print("\n" + "-" * 40)
print("\n[예시 2] 사전 검색 → 결과 분석 → 계산")
chain_steps_2 = [
    ("Dictionary", "LLM"),                          # 고정 입력
    ("StringProcessor", lambda prev: prev),          # 이전 결과를 입력으로
    ("Calculator", "7 * 8 + 6"),                    # 다시 고정 입력
]
tool_chain(registry, chain_steps_2)

[예시 1] 계산 결과를 문자열로 분석
=== Tool 체이닝 실행 ===

[Chain Step 1] Calculator
  입력: (100 + 200 + 300) / 3
  출력: 계산 결과: 200.0

[Chain Step 2] StringProcessor
  입력: 계산 결과: 200.0
  출력: 문자 수: 12, 단어 수: 3, 줄 수: 1

최종 결과: 문자 수: 12, 단어 수: 3, 줄 수: 1

----------------------------------------

[예시 2] 사전 검색 → 결과 분석 → 계산
=== Tool 체이닝 실행 ===

[Chain Step 1] Dictionary
  입력: LLM
  출력: LLM: 대규모 텍스트 데이터로 사전 학습된 거대 언어 모델

[Chain Step 2] StringProcessor
  입력: LLM: 대규모 텍스트 데이터로 사전 학습된 거대 언어 모델
  출력: 문자 수: 33, 단어 수: 9, 줄 수: 1

[Chain Step 3] Calculator
  입력: 7 * 8 + 6
  출력: 계산 결과: 62

최종 결과: 계산 결과: 62


'계산 결과: 62'

In [5]:
# ─────────────────────────────────────────
# 1번 파일에서 만든 ReActParser 재사용을 위해 import
# (같은 파일에 있다면 import 불필요)
# ─────────────────────────────────────────
import re

# ─────────────────────────────────────────
# simulate_react_with_tools
# 지금까지 만든 모든 것을 하나로 연결하는 함수
#
# question : 처음에 던지는 질문
# steps    : 하드코딩된 Thought-Action 시퀀스 (딕셔너리 리스트)
# registry : ToolRegistry 인스턴스
#
# 아직 LLM 없이 하드코딩된 steps를 사용하지만
# Tool 실행은 실제 Registry를 통해 이루어짐
# → "구조는 실제, 추론만 하드코딩" 상태
# ─────────────────────────────────────────
def simulate_react_with_tools(question, steps, registry):

    print(f"Question: {question}")
    print("=" * 50)

    for i, step in enumerate(steps, 1):
        print(f"\n--- Step {i} ---")
        print(f"Thought: {step['thought']}")

        # ─────────────────────────────────────────
        # Finish 액션이면 루프 종료
        # 최종 답변을 출력하고 함수 반환
        # ─────────────────────────────────────────
        if step['action'] == 'Finish':
            print(f"Action: Finish[{step['input']}]")
            print(f"\n{'=' * 50}")
            print(f"Final Answer: {step['input']}")
            return step['input']

        # ─────────────────────────────────────────
        # Finish가 아니면 Registry를 통해 Tool 실행
        # 실행 결과가 Observation이 됨
        # 실제 LLM 연동시 이 observation이
        # 다시 LLM 프롬프트에 추가되어 다음 Thought 생성
        # ─────────────────────────────────────────
        print(f"Action: {step['action']}[{step['input']}]")
        observation = registry.execute(step['action'], step['input'])
        print(f"Observation: {observation}")

    return None


# ─────────────────────────────────────────
# 시뮬레이션 실행
# steps : LLM이 생성해야 할 Thought-Action 시퀀스를
#         지금은 사람이 직접 작성
# ─────────────────────────────────────────
steps = [
    {
        "thought": "(100 + 250) * 0.1 의 값을 계산해야 합니다.",
        "action": "Calculator",       # Registry에서 찾을 Tool 이름
        "input": "(100 + 250) * 0.1"  # Tool에 넘길 입력값
    },
    {
        "thought": "계산 결과를 확인했습니다. 이제 ReAct에 대한 정의를 찾아보겠습니다.",
        "action": "Dictionary",
        "input": "ReAct"
    },
    {
        "thought": "필요한 정보를 모두 확인했습니다.",
        "action": "Finish",           # 루프 종료 신호
        "input": "(100+250)*0.1 = 35.0이고, ReAct는 추론과 행동을 결합한 프롬프팅 기법입니다."
    }
]

simulate_react_with_tools(
    "(100+250)*0.1의 값은 얼마이며, ReAct의 정의는 무엇인가?",
    steps,
    registry  # 3단계에서 만든 registry 재사용
)

Question: (100+250)*0.1의 값은 얼마이며, ReAct의 정의는 무엇인가?

--- Step 1 ---
Thought: (100 + 250) * 0.1 의 값을 계산해야 합니다.
Action: Calculator[(100 + 250) * 0.1]
Observation: 계산 결과: 35.0

--- Step 2 ---
Thought: 계산 결과를 확인했습니다. 이제 ReAct에 대한 정의를 찾아보겠습니다.
Action: Dictionary[ReAct]
Observation: ReAct: 추론(Reasoning)과 행동(Acting)을 결합한 프롬프팅 기법

--- Step 3 ---
Thought: 필요한 정보를 모두 확인했습니다.
Action: Finish[(100+250)*0.1 = 35.0이고, ReAct는 추론과 행동을 결합한 프롬프팅 기법입니다.]

Final Answer: (100+250)*0.1 = 35.0이고, ReAct는 추론과 행동을 결합한 프롬프팅 기법입니다.


'(100+250)*0.1 = 35.0이고, ReAct는 추론과 행동을 결합한 프롬프팅 기법입니다.'